# Clinical Scribe — Phase 1 (Free Colab T4)

Fine-tune **Qwen3-4B-Instruct-2507** with QLoRA to draft section-level clinical note text from doctor-patient dialogue.

> ⚠️ **Documentation-drafting aid, not a diagnostic tool.** Every output requires clinician review. Trained only on de-identified public benchmark data (MTS-Dialog, CC BY 4.0).

**This notebook is thin on purpose** — all logic lives in `src/clinical_scribe/`. It only: installs deps, mounts Drive, loads secrets, and calls the package via its CLI.

**Runtime:** set `Runtime → Change runtime type → T4 GPU` before running.

## 1. Get the code

In [ ]:
# Clone your repo onto the Colab VM (the kernel runs remotely, so your local
# files are NOT on it). Works from the VS Code Colab extension too.
import os, getpass
GITHUB_USER = "Nitheesh555"
REPO_NAME   = "clinical-scribe"
# Public repo -> just press Enter. Private repo -> paste a GitHub PAT (hidden).
# NOTE: google.colab `userdata` secrets only work in the Colab browser UI, not
# in VS Code, so we prompt instead.
_token = getpass.getpass("GitHub token (blank if repo is public): ").strip()
REPO_URL = (
    f"https://{_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    if _token else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
)
if not os.path.isdir(REPO_NAME):
    !git clone $REPO_URL
%cd $REPO_NAME
!ls

## 2. Install dependencies
Torch + CUDA are preinstalled on Colab — do **not** reinstall torch. Unsloth pulls compatible transformers/trl/peft/bitsandbytes.

In [ ]:
%pip install -q unsloth
%pip install -q -e ".[eval]"
# Capture the exact resolved versions for reproducibility.
!pip freeze > requirements.lock
print("Wrote requirements.lock")

## 3. Mount Drive (crash-resilient checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/clinical-scribe/outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Checkpoints ->', DRIVE_DIR)

## 4. Secrets
Add `HF_TOKEN` (and optionally `WANDB_API_KEY`) in the Colab **🔑 Secrets** panel. They are exported to the environment — never hardcoded or printed.

In [ ]:
from google.colab import userdata
for key in ('HF_TOKEN', 'WANDB_API_KEY'):
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f'{key}: set')
    except Exception:
        print(f'{key}: not provided (skipping)')

## 5. GPU check

In [ ]:
from clinical_scribe.utils import setup_logging, detect_gpu, log_versions
setup_logging()
detect_gpu()
log_versions()

## 6. Prepare data (MTS-Dialog → JSONL)

In [ ]:
!python scripts/prepare_data.py --config configs/phase1_t4.yaml --stats-out outputs/data_stats.json
import json; print(json.dumps(json.load(open('outputs/data_stats.json')), indent=2))

## 7. Train (QLoRA SFT)
Checkpoints to `general.checkpoint_dir` (set to Drive below). Re-run with `--resume` after a disconnect.

In [ ]:
# Point checkpoint mirroring at Drive without editing the YAML.
import yaml
with open('configs/_colab_override.yaml', 'w') as f:
    yaml.safe_dump({'extends': 'phase1_t4.yaml', 'general': {'checkpoint_dir': DRIVE_DIR}}, f)
!python -m clinical_scribe train --config configs/_colab_override.yaml  # add --resume to continue

## 8. Evaluate & export
**Evaluate** runs base-vs-fine-tuned (ROUGE-L / BERTScore / structure validity / faithfulness) and writes `outputs/eval_report.md` plus the Phase 1 **success gate** — STOP here and report numbers before Phase 2.

**Export** merges the LoRA adapter, writes a clinical model card, and (optionally) pushes to the HF Hub. Set `export.hub_repo_id` in the config to push; leave it unset for a local-only merge. GGUF needs a llama.cpp checkout (`export.export_gguf` + `export.llama_cpp_dir`).

In [ ]:
# Evaluate the fine-tuned adapter against the base model (writes eval_report.md + gate).
!python -m clinical_scribe evaluate --config configs/_colab_override.yaml --adapter outputs/adapter

# Merge + model card (+ Hub push if export.hub_repo_id is set). HF_TOKEN must be in the env.
!python -m clinical_scribe export --config configs/_colab_override.yaml --adapter outputs/adapter

## 9. Test / Demo (interactive)

Load the **merged** model from step 8 (`outputs/merged`, falling back to `export.hub_repo_id` if the local folder is absent) and serve a Gradio UI with a public share link.

Uses the package's own `build_messages` / `decorate_draft` so the prompt matches training **exactly** — no drift. Requires a **GPU runtime**.

> ⚠️ `launch(share=True)` blocks the notebook — **stop this cell (■) when you're done** before running anything else. The `*.gradio.live` link is for testing only and expires when the runtime stops.

In [ ]:
# ── 9. Test / Demo: load the merged model and serve a Gradio UI ───────────────
%pip install -q gradio

import torch, gradio as gr
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from clinical_scribe.config import load_config
from clinical_scribe.serve import build_messages, decorate_draft
from clinical_scribe.data import SECTION_LABELS

cfg = load_config("configs/_colab_override.yaml")

# Prefer the local merged folder from step 8; fall back to the Hub repo.
MERGED_DIR = Path(cfg.general.output_dir) / cfg.export.merged_subdir   # outputs/merged
MODEL_SRC = str(MERGED_DIR) if MERGED_DIR.exists() else cfg.export.hub_repo_id
assert MODEL_SRC, "No merged model found — run section 8 (export), or set export.hub_repo_id."
print("Loading model from:", MODEL_SRC)

tok = AutoTokenizer.from_pretrained(MODEL_SRC)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_SRC, dtype=torch.bfloat16, device_map="cuda").eval()
print("Loaded on:", next(model.parameters()).device)

@torch.inference_mode()
def generate(transcript: str, section: str) -> str:
    if not transcript.strip():
        return "Please paste a doctor–patient transcript."
    messages = build_messages(cfg, transcript, section)          # exact training prompt
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(
        inputs,
        max_new_tokens=cfg.serve.max_new_tokens,                 # 512
        do_sample=cfg.serve.temperature > 0,                     # temp 0.0 -> greedy
        pad_token_id=tok.pad_token_id,
    )
    text = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    return decorate_draft(text, cfg)                             # adds the DRAFT banner

SECTIONS = sorted(set(SECTION_LABELS.values()))
EXAMPLE = """Doctor: What brings you in today?
Patient: I've had chest pain for the past two days. It's a sharp pain on the left side.
Doctor: Does it radiate anywhere, like your arm or jaw?
Patient: Sometimes to my left shoulder.
Doctor: Any shortness of breath or sweating?
Patient: Yes, I get short of breath when I walk up stairs."""

with gr.Blocks(title="Clinical Scribe Demo") as demo:
    gr.Markdown("## Clinical Scribe Demo\nDocumentation aid only — not a diagnostic tool. All outputs require clinician review.")
    with gr.Row():
        with gr.Column():
            t = gr.Textbox(label="Doctor-Patient Transcript", lines=14, value=EXAMPLE)
            s = gr.Dropdown(SECTIONS, label="Section to generate", value="History of Present Illness")
            btn = gr.Button("Submit", variant="primary")
        out = gr.Textbox(label="Generated Note Section", lines=14)
    btn.click(generate, [t, s], out)

demo.launch(share=True)